# Lekcja 08 - Wzorzec Projektowy Multi-Agent


## Konfiguracja


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Dlaczego systemy wieloagentowe?

Zadania ze świata rzeczywistego, takie jak planowanie podróży, obejmują wiele różnych rodzajów wiedzy — logistykę, znajomość lokalną, budżetowanie i inne. Pojedynczy agent próbujący ogarnąć wszystko szybko staje się nieporęczny.

Systemy wieloagentowe rozwiązują to poprzez **specjalizację**: każdy agent skupia się na jednej dziedzinie wiedzy, dostarczając wyniki wyższej jakości niż ogólnikowiec. Poprawiają też **skalowalność** — można dodawać nowych agentów (np. specjalistę od lotów, krytyka restauracji) bez przepisywania istniejącego przepływu pracy. Agenci łączą się ze sobą w uporządkowanym procesie, przekazując kontekst od jednego do drugiego.


## Tworzenie Specjalistycznych Agentów


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Tworzenie sekwencyjnego przepływu pracy

`WorkflowBuilder` umożliwia łączenie agentów w graf skierowany. Tutaj tworzymy prostą dwuetapową linię procesu: **TravelPlanner** tworzy wstępny plan podróży, a następnie **TravelConcierge** go przegląda i udoskonala.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Dodawanie większej liczby agentów do przepływu pracy

Jedną z największych zalet wzorca wieloagentowego jest łatwość jego rozszerzania. Poniżej dodajemy agenta **BudgetReviewer**, który sprawdza plan w odniesieniu do budżetu podróżnego, oznacza elementy, które mogą przekroczyć limit kosztów, oraz sugeruje alternatywy pozwalające zaoszczędzić pieniądze. Przepływ pracy teraz uruchamia trzech agentów po kolei:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Podsumowanie

W tej lekcji nauczyłeś się, jak:

1. **Tworzyć wyspecjalizowanych agentów** — każdy z określoną rolą (planowanie, konsjerż, przegląd budżetu).
2. **Łączyć agentów w sekwencyjny przepływ pracy** za pomocą `WorkflowBuilder` i `add_edge`.
3. **Strumieniować wyjście** z wieloagentowego procesu, śledząc, który agent aktualnie mówi.
4. **Rozszerzać przepływ pracy** poprzez dodawanie nowych agentów do łańcucha bez modyfikowania istniejących.

Wzorzec projektowy wieloagentowy utrzymuje każdego agenta prostym, jednocześnie produkując bogatsze, bardziej gruntownie zweryfikowane wyniki niż jakikolwiek pojedynczy agent mógłby osiągnąć samodzielnie.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:  
Niniejszy dokument został przetłumaczony przy użyciu usługi tłumaczenia AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mimo że dokładamy wszelkich starań, aby zapewnić poprawność, prosimy pamiętać, że automatyczne tłumaczenia mogą zawierać błędy lub nieścisłości. Oryginalny dokument w języku źródłowym należy traktować jako źródło autorytatywne. W przypadku informacji krytycznych zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z korzystania z tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
